# Submission 5: LightGBM stacker with class-distance and local annotation-bias features

**Leaderboard-performance model. Heavily ID-derived.** Its inputs include the plain ID prior, SID, the XLM-R ID-head, the labels of neighbouring training rows by id, id-window class shares, the distance to the nearest labelled row of each class, and the raw id. These features are **dataset-specific and not proposed for deployment**. The team's genuine text methodology is reported separately.

**Validation class: STANDARD_OOF_STACKING_WITH_DEPENDENCY.** A row's own label and its own fold's labels never enter its features. The base OOF arrays and neighbour pools of the stacker's training rows do include the held-out fold's labels, and the class correction has one level of nesting.

**What is new over Submission 4 (18 features):**
- **(A)** Log id-distance to the nearest labelled row of each class: before, after, and the minimum of the two.
- **(B)** Local annotation bias: the mean over labelled rows within ±10/30/100 ids of (one-hot label − mean of the E5 and TF-IDF out-of-fold probabilities).

**Honest grouped OOF macro-F1:** 0.706268, against Submission 4's 0.704464 in the same environment. No leaderboard feedback and no external labelled data were used.

Run all cells. It takes about 4 minutes on 2 CPU cores and writes `uap_submission/submission/submission_5.csv`.

In [ ]:
import json, hashlib, re, time, platform
from pathlib import Path
import numpy as np, pandas as pd, lightgbm as lgb, sklearn
T0 = time.time(); HERE = Path.cwd(); ART = HERE / "artifacts_submission_5"
CFG = json.loads((ART / "config_submission_5.json").read_text())
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def find(n):
    for b in (HERE, HERE.parent, HERE.parent.parent):
        if (b / n).exists(): return b / n
    raise FileNotFoundError(n)
print("versions", dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__, lightgbm=lgb.__version__))
for n, h in CFG["data"].items(): assert sha(find(n)) == h, n
for n, h in CFG["artifacts"].items(): assert sha(ART / n) == h, n
tr = pd.read_csv(find("train.csv")); te = pd.read_csv(find("test.csv")); ss = pd.read_csv(find("sample_submission.csv"))
y = tr.y.to_numpy(np.int64); ids = tr.id.to_numpy(np.int64); tids = te.id.to_numpy(np.int64)
folds = np.load(ART / "fold_ids.npy").astype(np.int64)
assert hashlib.sha256(folds.tobytes()).hexdigest() == CFG["fold_int64_sha256"] and np.bincount(folds).tolist() == [9543, 9546, 9571, 9581, 9576]
assert np.array_equal(tids, ss.id.to_numpy())
SRC = {n: np.load(ART / f"src_{n}.npz") for n in ["tfidf", "knn", "e5", "sid", "pid", "idh"]}
for n, z in SRC.items(): assert z["oof"].shape == (len(y), 3) and z["test"].shape == (len(tids), 3) and np.isfinite(z["oof"]).all()
def mf1(yt, p):
    cm = np.bincount(yt * 3 + p, minlength=9).reshape(3, 3); tp = np.diag(cm).astype(float)
    P = tp / np.maximum(cm.sum(0), 1); R = tp / np.maximum(cm.sum(1), 1); return float((2 * P * R / np.maximum(P + R, 1e-12)).mean())
def f1s(yt, p):
    cm = np.bincount(yt * 3 + p, minlength=9).reshape(3, 3); tp = np.diag(cm).astype(float)
    P = tp / np.maximum(cm.sum(0), 1); R = tp / np.maximum(cm.sum(1), 1); return 2 * P * R / np.maximum(P + R, 1e-12)
def nb_feats(q, pool_ids, pool_y):
    o = np.argsort(pool_ids); s = pool_ids[o]; sy = pool_y[o]; n = len(s); pos = np.searchsorted(s, q); F = {}
    with np.errstate(invalid="ignore", divide="ignore"):
        for i in (1, 2, 3):
            p = pos - i; ok = p >= 0; pc = np.clip(p, 0, n - 1)
            F[f"prev{i}_y"] = np.where(ok, sy[pc], -1); F[f"prev{i}_d"] = np.where(ok, np.log1p(q - s[pc]), 9.0)
            nx = pos + i - 1; ok = nx < n; nc = np.clip(nx, 0, n - 1)
            F[f"next{i}_y"] = np.where(ok, sy[nc], -1); F[f"next{i}_d"] = np.where(ok, np.log1p(s[nc] - q), 9.0)
    cum = np.vstack([np.zeros((1, 3)), np.cumsum(np.eye(3)[sy], 0)])
    for Wn in (3, 10, 30, 100, 300):
        a = np.searchsorted(s, q - Wn, "left"); b = np.searchsorted(s, q + Wn, "right"); c = cum[b] - cum[a]; t = c.sum(1)
        F[f"w{Wn}_n"] = t
        for k in range(3): F[f"w{Wn}_p{k}"] = np.where(t > 0, c[:, k] / np.maximum(t, 1), np.nan)
    return pd.DataFrame(F)
def text_meta(txt):
    t = pd.Series(txt).astype(str); L = t.str.len()
    return pd.DataFrame({"len": L, "nwords": t.str.split().str.len(), "bn": t.str.contains("[ঀ-৿]").astype(int),
        "lat": t.str.contains("[A-Za-z]").astype(int), "emoji": t.str.count("[\U0001F300-\U0001FAFF☀-➿]"),
        "caps": t.str.count("[A-Z]") / L.clip(lower=1), "excl": t.str.count("!"), "q": t.str.count(r"\?"), "digits": t.str.count(r"\d") / L.clip(lower=1)})
E5o, TFo = SRC["e5"]["oof"], SRC["tfidf"]["oof"]
def extra_feats(q, pool):
    """NEW in Submission 5. (A) log id-distance to the nearest labelled pool row of each class, before/after/min.
    (B) local annotation bias: mean over pool rows within +-W ids of (one-hot label - mean(E5, TF-IDF) OOF probs)."""
    pid_ = ids[pool]; o = np.argsort(pid_); s = pid_[o]; sy = y[pool][o]; F = {}
    with np.errstate(divide="ignore", invalid="ignore"):
        for c in range(3):
            sc = s[sy == c]; n = len(sc); pos = np.searchsorted(sc, q)
            pv = np.where(pos > 0, q - sc[np.clip(pos - 1, 0, n - 1)], 10 ** 6); nx = np.where(pos < n, sc[np.clip(pos, 0, n - 1)] - q, 10 ** 6)
            F[f"cd{c}_prev"] = np.log1p(pv); F[f"cd{c}_next"] = np.log1p(nx); F[f"cd{c}_min"] = np.log1p(np.minimum(pv, nx))
    R = (np.eye(3)[sy] - 0.5 * (E5o[pool][o] + TFo[pool][o])); cum = np.vstack([np.zeros((1, 3)), np.cumsum(R, 0)])
    for W in (10, 30, 100):
        a = np.searchsorted(s, q - W, "left"); b = np.searchsorted(s, q + W, "right"); t = np.maximum(b - a, 1)
        for c in range(3): F[f"bias{W}_{c}"] = (cum[b][:, c] - cum[a][:, c]) / t
    return pd.DataFrame(F)
def probs(split): return pd.DataFrame({f"{n}_p{k}": SRC[n]["oof" if split == "tr" else "test"][:, k] for n in SRC for k in range(3)})
parts, eparts = [], []
for k in range(5):
    va = np.flatnonzero(folds == k); pool = np.flatnonzero(folds != k)     # a row's own fold is never in its pool
    f = nb_feats(ids[va], ids[pool], y[pool]); f.index = va; parts.append(f)
    e = extra_feats(ids[va], pool); e.index = va; eparts.append(e)
Ftr = pd.concat([probs("tr"), pd.concat(parts).sort_index().reset_index(drop=True), text_meta(tr.text)], axis=1); Ftr["rid"] = ids
Fte0 = pd.concat([probs("te"), nb_feats(tids, ids, y), text_meta(te.text)], axis=1); Fte0["rid"] = tids
X = pd.concat([Ftr, pd.concat(eparts).sort_index().reset_index(drop=True)], axis=1)
assert list(X.columns) == CFG["feature_order"], "feature order changed"
nbcols = [c for c in Ftr.columns if c.startswith(("prev", "next", "w"))]
cw = len(y) / (3 * np.bincount(y)); wts = cw[y]
OOF = np.zeros((len(y), 3)); TST = np.zeros((len(tids), 3)); seeds = {}
for SEED in CFG["seeds"]:
    params = dict(CFG["lightgbm"], seed=SEED, bagging_seed=SEED, feature_fraction_seed=SEED)
    oof = np.zeros((len(y), 3)); tst = np.zeros((len(tids), 3))
    for k in range(5):
        trm = folds != k; va = folds == k
        m = lgb.train(params, lgb.Dataset(X.loc[trm], y[trm], weight=wts[trm]), CFG["rounds"]); oof[va] = m.predict(X.loc[va])
        pool = np.flatnonzero(folds != k)                                  # test features use the same pool density
        XT = pd.concat([Fte0.copy(), extra_feats(tids, pool)], axis=1); XT[nbcols] = nb_feats(tids, ids[pool], y[pool])[nbcols].values
        tst += m.predict(XT[X.columns]) / 5
    seeds[SEED] = round(mf1(y, oof.argmax(1)), 6); print(f"seed {SEED}: uncorrected OOF {seeds[SEED]} ({time.time() - T0:.0f}s)", flush=True)
    OOF += oof / 3; TST += tst / 3
L = np.log(np.clip(OOF, 1e-9, 1)); GRID = [np.array([0, a, b]) for a in np.arange(-.5, .51, .05) for b in np.arange(-1.2, .21, .05)]
pred = np.zeros(len(y), int); OFF = []
for k in range(5):                     # correction for fold k chosen on the other four folds only
    t = folds != k; b = max(GRID, key=lambda b: mf1(y[t], (L[t] + b).argmax(1))); pred[folds == k] = (L[folds == k] + b).argmax(1); OFF.append(b)
FIN = np.median(np.array(OFF), 0); fs = [mf1(y[folds == k], pred[folds == k]) for k in range(5)]
RES = dict(oof=round(mf1(y, pred), 6), per_class=f1s(y, pred).round(6).tolist(), accuracy_not_f1=round(float((pred == y).mean()), 6),
           confusion=np.bincount(y * 3 + pred, minlength=9).reshape(3, 3).tolist(), folds=np.round(fs, 6).tolist(), fold_std=round(float(np.std(fs)), 6),
           worst=round(min(fs), 6), seeds=seeds, fold_offsets=[o.round(2).tolist() for o in OFF], final_offset=FIN.round(3).tolist())
print(json.dumps(RES))
assert abs(RES["oof"] - CFG["audited_oof"]) <= 0.001, "OOF differs from the audited value by more than 0.001"
tp = (np.log(np.clip(TST, 1e-9, 1)) + FIN).argmax(1)
share = np.bincount(tp, minlength=3) / len(tp); assert (np.abs(share - np.bincount(y, minlength=3) / len(y)) <= 0.06).all()
URL = re.compile(r"(?:https?://|www\.)\S+", re.I); MEN = re.compile(r"@\w+"); WS = re.compile(r"\s+", re.UNICODE)
clean = lambda s: WS.sub(" ", MEN.sub(" USER ", URL.sub(" URL ", str(s)))).strip()
ctr = tr.text.astype(str).map(clean).to_numpy(); cte = te.text.astype(str).map(clean).to_numpy()
gg = pd.DataFrame({"t": pd.Series(list(ctr), dtype="object"), "y": y}).groupby("t")["y"].agg(["nunique", "first"])
LK = {str(t): int(v) for t, v in gg[gg["nunique"] == 1]["first"].items()}
final = tp.copy(); hits = 0
for i, t in enumerate(cte):
    v = LK.get(str(t))
    if v is not None: final[i] = v; hits += 1
print(f"test share {share.round(4).tolist()} | duplicate override {hits} matched, {int((final != tp).sum())} changed")
def validate(df):
    assert list(df.columns) == ["id", "y_pred"] and len(df) == 11955 and df.id.is_unique and np.array_equal(df.id.to_numpy(), ss.id.to_numpy())
    assert not df.isna().any().any() and df.y_pred.dtype.kind in "iu" and set(df.y_pred.unique()) <= {0, 1, 2}
sub = pd.DataFrame({"id": tids, "y_pred": final.astype(int)}); validate(sub)
OUT = next((b / "uap_submission" / "submission" for b in [HERE] + list(HERE.parents) if (b / "uap_submission" / "submission").is_dir()), HERE)
CSV = OUT / "submission_5.csv"; sub.to_csv(CSV, index=False); validate(pd.read_csv(CSV))
print("wrote", CSV, "SHA256", sha(CSV), "| audited", CFG.get("audited_csv_sha256"), f"| {time.time() - T0:.0f}s | KAGGLE UPLOAD: NO")